
# Major Index DL One-At-A-Time Sensitivity

This notebook performs the post-selection sensitivity stage for the deep-learning branch workflow.

For each `model x horizon` setup:

- the best full-span branch/config pair from notebook `14` is taken as the base
- ten one-at-a-time hyperparameter alterations are generated from the DL grid
- the base and all one-at-a-time variants are rerun on the full rolling span
- the resulting deltas are summarized in tuning and pure-test space

The branch selected by notebook `14` is held fixed during the sensitivity reruns.


In [ ]:

from __future__ import annotations

import json
import random
import time
from contextlib import contextmanager
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from qiskit.circuit.library import ZZFeatureMap
try:
    from qiskit.circuit.library import zz_feature_map
except Exception:
    zz_feature_map = None
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error
import mthree


def resolve_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Run this notebook from the project root or its notebooks directory.')


PROJECT_ROOT = resolve_project_root()
SELECTION_PATH = PROJECT_ROOT / 'outputs' / '14_major_index_dl_top5_branch_comparison' / 'tables' / 'best_branch_config_by_setup.csv'
PIPELINE_BUNDLE_PATH = PROJECT_ROOT / 'outputs' / '01_major_index_sliding_window_pipeline' / 'major_index_sliding_window_bundle.joblib'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / '15_major_index_dl_branch_sensitivity'
LOG_DIR = OUTPUT_ROOT / 'rolling_logs'
PREDICTIONS_DIR = OUTPUT_ROOT / 'predictions'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIG_DIR = OUTPUT_ROOT / 'figures'
for directory in [OUTPUT_ROOT, LOG_DIR, PREDICTIONS_DIR, TABLE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ['rsi14', 'macd_hist', 'bb_z20', 'roc10', 'vol20', 'sma10_gap', 'atr14_pct', 'williams_r14']
TARGET_COL = 'target_close'
RANDOM_SEED = 42
NUM_QUBITS = len(FEATURE_COLS)
STATEVECTOR_BATCH_SIZE = 16
SHOTS = 128
ZNE_FOLD_SCALES = [1, 3, 5]
FORCE_FRESH_RUN = False
FLUSH_EVERY_N_ROWS = 1

DL_GRID = {
    'hidden_dim': [32, 64, 128],
    'num_layers': [1, 2, 3],
    'learning_rate': [1e-3, 3e-4, 1e-4],
    'activation': ['relu', 'tanh', 'silu'],
    'weight_decay': [0.0, 1e-4, 1e-3],
    'dropout': [0.0],
    'batch_size': [16],
    'epoch': [32],
}
BRANCH_TO_MODE = {
    'quantum_ideal': 'ideal',
    'quantum_aer_noisy': 'aer_noisy',
    'quantum_mthree_readout': 'mthree_readout',
    'quantum_zne': 'zne',
}

ALL_LOGS_PATH = OUTPUT_ROOT / 'all_sensitivity_logs.csv'
VARIANT_MANIFEST_PATH = TABLE_DIR / 'variant_manifest.csv'
SCORES_SUMMARY_PATH = TABLE_DIR / 'scores_summary.csv'
SETUP_SUMMARY_PATH = TABLE_DIR / 'setup_summary.csv'
DELTA_SUMMARY_PATH = TABLE_DIR / 'delta_summary.csv'
TIMING_SUMMARY_PATH = TABLE_DIR / 'timing_summary.csv'

LOG_DATE_COLS = ['train_end_date', 'test_reference_date', 'test_target_date']
WINDOW_KEY_COLS = ['branch_name', 'setup_key', 'case_key', 'variant_index', 'window_id']
TIMING_ROWS: list[dict] = []

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')


In [ ]:

@contextmanager
def timed_step(step_name: str):
    start = time.perf_counter()
    yield
    elapsed_seconds = time.perf_counter() - start
    TIMING_ROWS.append({'step': step_name, 'elapsed_seconds': elapsed_seconds})
    print(f'[TIMER] {step_name}: {elapsed_seconds:.2f}s')



def set_seeds(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



def load_case_bundle(path: Path = PIPELINE_BUNDLE_PATH) -> dict:
    bundle = joblib.load(path)
    return bundle['case_bundle']



def reset_output_targets() -> None:
    for path in [ALL_LOGS_PATH, VARIANT_MANIFEST_PATH, SCORES_SUMMARY_PATH, SETUP_SUMMARY_PATH, DELTA_SUMMARY_PATH, TIMING_SUMMARY_PATH]:
        if path.exists():
            path.unlink()
    for directory in [LOG_DIR, PREDICTIONS_DIR, FIG_DIR]:
        for csv_path in directory.glob('*.csv'):
            csv_path.unlink()
        for png_path in directory.glob('*.png'):
            png_path.unlink()



def append_frame(path: Path, frame: pd.DataFrame) -> None:
    if frame is None or frame.empty:
        return
    frame.to_csv(path, mode='a', header=not path.exists(), index=False)



def load_progress_frame(path: Path = ALL_LOGS_PATH) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    for col in LOG_DATE_COLS:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors='coerce')
    if 'is_tuning_window' in frame.columns:
        frame['is_tuning_window'] = frame['is_tuning_window'].astype(str).str.lower().eq('true')
    if all(col in frame.columns for col in WINDOW_KEY_COLS):
        frame = frame.drop_duplicates(subset=WINDOW_KEY_COLS, keep='last').reset_index(drop=True)
    return frame



def window_result_key(branch_name: str, setup_key: str, case_key: str, variant_index: int, window_id: int) -> tuple[str, str, str, int, int]:
    return (str(branch_name), str(setup_key), str(case_key), int(variant_index), int(window_id))



def completed_window_keys(frame: pd.DataFrame) -> set[tuple[str, str, str, int, int]]:
    if frame.empty or 'status' not in frame.columns:
        return set()
    completed = frame.loc[frame['status'].eq('completed')].copy()
    if completed.empty:
        return set()
    return {
        window_result_key(row['branch_name'], row['setup_key'], row['case_key'], int(row['variant_index']), int(row['window_id']))
        for _, row in completed.iterrows()
    }



def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt = np.asarray(y_true, dtype=float)[mask]
    yp = np.asarray(y_pred, dtype=float)[mask]
    if len(yt) == 0:
        return {'MAE': np.nan, 'RMSE': np.nan, 'R2': np.nan, 'n_points': 0}
    return {
        'MAE': float(mean_absolute_error(yt, yp)),
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'R2': float(r2_score(yt, yp)) if len(yt) > 1 else np.nan,
        'n_points': int(len(yt)),
    }



def add_error_columns(frame: pd.DataFrame) -> pd.DataFrame:
    if frame is None or len(frame) == 0:
        return frame.copy() if isinstance(frame, pd.DataFrame) else pd.DataFrame()
    enriched = frame.copy()
    enriched['error_points'] = enriched['predicted_close'] - enriched['actual_close']
    enriched['error_pct'] = np.where(
        np.isfinite(enriched['actual_close']) & (np.abs(enriched['actual_close']) > 1e-12),
        100.0 * enriched['error_points'] / enriched['actual_close'],
        np.nan,
    )
    enriched['abs_error_pct'] = np.abs(enriched['error_pct'])
    enriched['analysis_partition'] = np.where(enriched['is_tuning_window'], 'hp_tuning', 'pure_test')
    return enriched


In [ ]:

def fit_quantum_preprocessor(X_train: np.ndarray) -> dict:
    scaler = StandardScaler().fit(X_train)
    X_scaled = scaler.transform(X_train)
    mins = X_scaled.min(axis=0)
    maxs = X_scaled.max(axis=0)
    spans = np.where((maxs - mins) < 1e-8, 1.0, (maxs - mins))
    return {'scaler': scaler, 'mins': mins, 'spans': spans}


def transform_to_angles(bundle: dict, X: np.ndarray) -> np.ndarray:
    X_scaled = bundle['scaler'].transform(X)
    angles = np.pi * (X_scaled - bundle['mins']) / bundle['spans']
    angles = np.nan_to_num(angles, nan=0.0, posinf=np.pi, neginf=0.0)
    return np.clip(angles, 0.0, np.pi).astype(np.float32)


def build_feature_map(feature_dimension: int, reps: int):
    if zz_feature_map is not None:
        return zz_feature_map(feature_dimension=feature_dimension, reps=int(reps), entanglement='linear')
    return ZZFeatureMap(feature_dimension=feature_dimension, reps=int(reps), entanglement='linear')


def choose_anchor_indices(n_train: int, anchor_count: int) -> np.ndarray:
    if anchor_count >= n_train:
        return np.arange(n_train, dtype=np.int32)
    return np.unique(np.linspace(0, n_train - 1, num=int(anchor_count), dtype=np.int32))


def compute_statevector_anchor_features(angle_matrix: np.ndarray, anchor_angle_matrix: np.ndarray, reps: int) -> np.ndarray:
    feature_map = build_feature_map(angle_matrix.shape[1], reps=reps)
    anchor_statevectors = []
    for row in anchor_angle_matrix:
        circuit = feature_map.assign_parameters(row.tolist(), inplace=False)
        anchor_statevectors.append(Statevector.from_instruction(circuit).data)
    anchor_statevectors = np.asarray(anchor_statevectors, dtype=np.complex128)
    rows = []
    for start in range(0, len(angle_matrix), STATEVECTOR_BATCH_SIZE):
        batch = angle_matrix[start : start + STATEVECTOR_BATCH_SIZE]
        batch_states = []
        for row in batch:
            circuit = feature_map.assign_parameters(row.tolist(), inplace=False)
            batch_states.append(Statevector.from_instruction(circuit).data)
        batch_states = np.asarray(batch_states, dtype=np.complex128)
        overlaps = batch_states @ anchor_statevectors.conj().T
        rows.append((np.abs(overlaps) ** 2).astype(np.float32))
    return np.vstack(rows)


def build_generic_noise_model(num_qubits: int, scale_factor: float = 1.0) -> NoiseModel:
    model = NoiseModel()
    readout_p = min(max(0.03 * float(scale_factor), 1e-6), 0.49)
    readout = ReadoutError([[1.0 - readout_p, readout_p], [readout_p, 1.0 - readout_p]])
    for qubit in range(int(num_qubits)):
        model.add_readout_error(readout, [qubit])
    error_1q = depolarizing_error(min(0.001 * float(scale_factor), 0.2), 1)
    error_2q = depolarizing_error(min(0.01 * float(scale_factor), 0.2), 2)
    for gate in ['rz', 'sx', 'x', 'p', 'h', 'u', 'u1', 'u2', 'u3']:
        try:
            model.add_all_qubit_quantum_error(error_1q, [gate])
        except Exception:
            pass
    for gate in ['cx', 'ecr', 'cz']:
        try:
            model.add_all_qubit_quantum_error(error_2q, [gate])
        except Exception:
            pass
    return model


def add_measurements(circuit):
    measured = circuit.copy()
    measured.measure_all()
    return measured


def counts_to_probability_vector(counts: dict, num_qubits: int) -> np.ndarray:
    vec = np.zeros(2 ** int(num_qubits), dtype=np.float64)
    total = max(float(sum(counts.values())), 1.0)
    for bitstring, count in counts.items():
        idx = int(str(bitstring).replace(' ', ''), 2)
        vec[idx] = float(count) / total
    denom = max(vec.sum(), 1e-12)
    return vec / denom


def distribution_to_probability_vector(distribution, num_qubits: int) -> np.ndarray:
    vec = np.zeros(2 ** int(num_qubits), dtype=np.float64)
    for state, value in dict(distribution).items():
        if isinstance(state, (int, np.integer)):
            idx = int(state)
        else:
            idx = int(str(state).replace(' ', ''), 2)
        vec[idx] = float(value)
    denom = max(vec.sum(), 1e-12)
    return vec / denom


def bhattacharyya_similarity(prob_matrix: np.ndarray, anchor_prob_matrix: np.ndarray) -> np.ndarray:
    similarities = np.sqrt(prob_matrix[:, None, :] * anchor_prob_matrix[None, :, :]).sum(axis=2)
    return np.clip(similarities ** 2, 0.0, 1.0)


def simulate_prob_features(angle_matrix: np.ndarray, anchor_angle_matrix: np.ndarray, reps: int, shots: int, mitigation_mode: str, noise_scale: float = 1.0) -> np.ndarray:
    feature_map = build_feature_map(angle_matrix.shape[1], reps=reps)
    noise_model = build_generic_noise_model(angle_matrix.shape[1], scale_factor=noise_scale)
    simulator = AerSimulator(noise_model=noise_model)
    def circuit_to_distribution(row: np.ndarray):
        circuit = feature_map.assign_parameters(row.tolist(), inplace=False)
        measured = add_measurements(circuit)
        compiled = transpile(measured, simulator, optimization_level=0)
        result = simulator.run(compiled, shots=int(shots)).result()
        counts = result.get_counts()
        if mitigation_mode == 'mthree_readout':
            measured_qubits = list(range(angle_matrix.shape[1]))
            mitigation = mthree.M3Mitigation(simulator)
            mitigation.cals_from_system(measured_qubits)
            quasi = mitigation.apply_correction(counts, measured_qubits)
            if hasattr(quasi, 'nearest_probability_distribution'):
                corrected = quasi.nearest_probability_distribution()
            elif hasattr(quasi, 'probabilities'):
                corrected = quasi.probabilities()
            else:
                corrected = dict(quasi)
            return distribution_to_probability_vector(corrected, angle_matrix.shape[1])
        return counts_to_probability_vector(counts, angle_matrix.shape[1])
    sample_prob = np.asarray([circuit_to_distribution(row) for row in angle_matrix], dtype=np.float64)
    anchor_prob = np.asarray([circuit_to_distribution(row) for row in anchor_angle_matrix], dtype=np.float64)
    return bhattacharyya_similarity(sample_prob, anchor_prob).astype(np.float32)


def zne_extrapolate(scales: list[int], values: np.ndarray) -> np.ndarray:
    out = np.zeros(values.shape[1:], dtype=np.float64)
    for i in range(values.shape[1]):
        for j in range(values.shape[2]):
            coeffs = np.polyfit(scales, values[:, i, j], 1)
            out[i, j] = np.polyval(coeffs, 0.0)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def prepare_quantum_features(X_train: np.ndarray, X_other: np.ndarray, quantum_params: dict) -> dict:
    bundle = fit_quantum_preprocessor(X_train)
    train_angles = transform_to_angles(bundle, X_train)
    other_angles = transform_to_angles(bundle, X_other)
    anchor_indices = choose_anchor_indices(len(X_train), int(quantum_params['anchor_count']))
    anchor_angles = train_angles[anchor_indices]
    reps = int(quantum_params['feature_map_reps'])
    shots = int(quantum_params['shots'])
    mitigation_mode = quantum_params['mitigation_mode']
    if mitigation_mode == 'ideal':
        train_anchor = compute_statevector_anchor_features(train_angles, anchor_angles, reps=reps)
        other_anchor = compute_statevector_anchor_features(other_angles, anchor_angles, reps=reps)
    elif mitigation_mode == 'aer_noisy':
        train_anchor = simulate_prob_features(train_angles, anchor_angles, reps=reps, shots=shots, mitigation_mode='aer_noisy', noise_scale=1.0)
        other_anchor = simulate_prob_features(other_angles, anchor_angles, reps=reps, shots=shots, mitigation_mode='aer_noisy', noise_scale=1.0)
    elif mitigation_mode == 'mthree_readout':
        train_anchor = simulate_prob_features(train_angles, anchor_angles, reps=reps, shots=shots, mitigation_mode='mthree_readout', noise_scale=1.0)
        other_anchor = simulate_prob_features(other_angles, anchor_angles, reps=reps, shots=shots, mitigation_mode='mthree_readout', noise_scale=1.0)
    elif mitigation_mode == 'zne':
        train_stack = []
        other_stack = []
        for scale in ZNE_FOLD_SCALES:
            train_stack.append(simulate_prob_features(train_angles, anchor_angles, reps=reps, shots=shots, mitigation_mode='aer_noisy', noise_scale=float(scale)))
            other_stack.append(simulate_prob_features(other_angles, anchor_angles, reps=reps, shots=shots, mitigation_mode='aer_noisy', noise_scale=float(scale)))
        train_anchor = zne_extrapolate(ZNE_FOLD_SCALES, np.stack(train_stack, axis=0))
        other_anchor = zne_extrapolate(ZNE_FOLD_SCALES, np.stack(other_stack, axis=0))
    else:
        raise ValueError(mitigation_mode)
    return {'train_anchor_features': train_anchor, 'other_anchor_features': other_anchor}


def _act(name: str):
    if name == 'relu':
        return nn.ReLU()
    if name == 'tanh':
        return nn.Tanh()
    if name == 'silu':
        return nn.SiLU()
    raise ValueError(name)


class TorchModel(nn.Module):
    def __init__(self, model_name: str, n_features: int, hp: dict):
        super().__init__()
        h = int(hp['hidden_dim'])
        n = int(hp['num_layers'])
        d = float(hp['dropout'])
        act = hp['activation']
        if model_name == 'MLP':
            blocks = []
            for i in range(n):
                in_dim = n_features if i == 0 else h
                blocks.extend([nn.Linear(in_dim, h), _act(act), nn.Dropout(d)])
            self.backbone = nn.Sequential(*blocks)
            self.head = nn.Linear(h, 1)
            self.kind = 'MLP'
        else:
            self.proj = nn.Linear(n_features, h)
            recurrent_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[model_name]
            self.rnn = recurrent_cls(input_size=h, hidden_size=h, num_layers=n, dropout=d if n > 1 else 0.0, batch_first=True)
            self.drop = nn.Dropout(d)
            self.head = nn.Linear(h, 1)
            self.kind = model_name
    def forward(self, x):
        if self.kind == 'MLP':
            return self.head(self.backbone(x)).squeeze(-1)
        z = self.proj(x).unsqueeze(1)
        out, _ = self.rnn(z)
        return self.head(self.drop(out[:, -1, :])).squeeze(-1)


def fit_predict_window_torch(model_name: str, hp: dict, X_tr_s: np.ndarray, y_tr_s: np.ndarray, X_te_s: np.ndarray) -> float:
    set_seeds(RANDOM_SEED)
    model = TorchModel(model_name, X_tr_s.shape[1], hp).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(hp['learning_rate']), weight_decay=float(hp['weight_decay']))
    loss_fn = nn.MSELoss()
    dataset = TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr_s, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=min(int(hp['batch_size']), len(dataset)), shuffle=False)
    model.train()
    for _ in range(int(hp['epoch'])):
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        xt = torch.tensor(X_te_s, dtype=torch.float32, device=DEVICE)
        pred = model(xt).detach().cpu().numpy().reshape(-1)
    return float(pred[0])


def prepare_model_inputs(anchor_features: np.ndarray, model_name: str) -> np.ndarray:
    if model_name == 'MLP':
        return anchor_features.astype(np.float32)
    return anchor_features[:, :, None].astype(np.float32)


def predict_one_window(branch_name: str, model_name: str, config: dict, train_df: pd.DataFrame, test_df: pd.DataFrame) -> float:
    y_train = train_df[TARGET_COL].to_numpy(dtype=float)
    y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    y_tr_scaled = y_scaler.transform(y_train.reshape(-1, 1)).ravel()
    if branch_name == 'conventional':
        x_scaler = StandardScaler().fit(train_df[FEATURE_COLS].to_numpy(dtype=float))
        X_tr_ready = x_scaler.transform(train_df[FEATURE_COLS].to_numpy(dtype=float)).astype(np.float32)
        X_te_ready = x_scaler.transform(test_df[FEATURE_COLS].to_numpy(dtype=float)).astype(np.float32)
        pred_scaled = fit_predict_window_torch(model_name, config, X_tr_ready, y_tr_scaled, X_te_ready)
        return float(y_scaler.inverse_transform(np.array(pred_scaled).reshape(-1, 1)).ravel()[0])
    quantum_params = {'feature_map_reps': 1, 'anchor_count': 6, 'shots': SHOTS, 'mitigation_mode': BRANCH_TO_MODE[branch_name]}
    X_train = train_df[FEATURE_COLS].to_numpy(dtype=float)
    X_test = test_df[FEATURE_COLS].to_numpy(dtype=float)
    prepared = prepare_quantum_features(X_train, X_test, quantum_params)
    X_tr_ready = prepare_model_inputs(prepared['train_anchor_features'], model_name)
    X_te_ready = prepare_model_inputs(prepared['other_anchor_features'], model_name)
    pred_scaled = fit_predict_window_torch(model_name, config, X_tr_ready, y_tr_scaled, X_te_ready)
    return float(y_scaler.inverse_transform(np.array(pred_scaled).reshape(-1, 1)).ravel()[0])


In [ ]:

def load_best_setup(path: Path = SELECTION_PATH) -> pd.DataFrame:
    best = pd.read_csv(path)
    best['setup_key'] = best['model_name'] + '_w' + best['window_size'].astype(str)
    return best.sort_values(['model_name', 'window_size']).reset_index(drop=True)


def build_variant_manifest(best_setup: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in best_setup.iterrows():
        base_config = json.loads(row['config_json'])
        setup_key = row['setup_key']
        branch_name = row['branch_name']
        model_name = row['model_name']
        rows.append({
            'setup_key': setup_key,
            'model_name': model_name,
            'window_size': int(row['window_size']),
            'branch_name': branch_name,
            'variant_index': 0,
            'variant_kind': 'base',
            'variant_label': 'base',
            'changed_param': 'none',
            'changed_to': 'base',
            'config_json': json.dumps(base_config, sort_keys=True),
            'config_summary': row['effective_config_summary'],
            'base_config_key': row['config_key'],
        })
        variant_index = 1
        for param in ['hidden_dim', 'num_layers', 'learning_rate', 'activation', 'weight_decay']:
            current = base_config[param]
            for alt in DL_GRID[param]:
                if alt == current:
                    continue
                variant = dict(base_config)
                variant[param] = alt
                rows.append({
                    'setup_key': setup_key,
                    'model_name': model_name,
                    'window_size': int(row['window_size']),
                    'branch_name': branch_name,
                    'variant_index': variant_index,
                    'variant_kind': 'oat',
                    'variant_label': f'{param}__{alt}',
                    'changed_param': param,
                    'changed_to': str(alt),
                    'config_json': json.dumps(variant, sort_keys=True),
                    'config_summary': json.dumps(variant, sort_keys=True),
                    'base_config_key': row['config_key'],
                })
                variant_index += 1
    manifest = pd.DataFrame(rows)
    return manifest.sort_values(['setup_key', 'variant_index']).reset_index(drop=True)


def metric_block(frame: pd.DataFrame, prefix: str) -> dict:
    if frame.empty:
        return {f'{prefix}_close_MAE': np.nan, f'{prefix}_close_RMSE': np.nan, f'{prefix}_close_R2': np.nan, f'{prefix}_close_n': 0, f'{prefix}_abs_error_pct_mean': np.nan, f'{prefix}_error_pct_bias': np.nan, f'{prefix}_windows': 0}
    metrics = regression_metrics(frame['actual_close'].to_numpy(dtype=float), frame['predicted_close'].to_numpy(dtype=float))
    return {f'{prefix}_close_MAE': metrics['MAE'], f'{prefix}_close_RMSE': metrics['RMSE'], f'{prefix}_close_R2': metrics['R2'], f'{prefix}_close_n': metrics['n_points'], f'{prefix}_abs_error_pct_mean': float(frame['abs_error_pct'].mean()), f'{prefix}_error_pct_bias': float(frame['error_pct'].mean()), f'{prefix}_windows': int(len(frame))}


def build_scores_summary(logs: pd.DataFrame) -> pd.DataFrame:
    if logs.empty:
        return pd.DataFrame()
    rows = []
    group_cols = ['setup_key', 'branch_name', 'case_key', 'index_key', 'window_size', 'model_name', 'variant_index', 'variant_kind', 'variant_label', 'changed_param', 'changed_to', 'config_json', 'base_config_key']
    for keys, block in logs.groupby(group_cols, dropna=False):
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update(metric_block(block, 'full'))
        row.update(metric_block(block.loc[block['analysis_partition'].eq('hp_tuning')], 'tuning'))
        row.update(metric_block(block.loc[block['analysis_partition'].eq('pure_test')], 'pure_test'))
        row['mean_fit_seconds'] = float(block['fit_seconds'].mean())
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['setup_key', 'case_key', 'variant_index']).reset_index(drop=True)


def build_setup_summary(scores: pd.DataFrame) -> pd.DataFrame:
    if scores.empty:
        return pd.DataFrame()
    group_cols = ['setup_key', 'branch_name', 'window_size', 'model_name', 'variant_index', 'variant_kind', 'variant_label', 'changed_param', 'changed_to', 'config_json', 'base_config_key']
    agg = scores.groupby(group_cols, as_index=False).agg(
        avg_tuning_abs_error_pct_mean=('tuning_abs_error_pct_mean', 'mean'),
        avg_pure_test_abs_error_pct_mean=('pure_test_abs_error_pct_mean', 'mean'),
        avg_pure_test_close_RMSE=('pure_test_close_RMSE', 'mean'),
        avg_full_abs_error_pct_mean=('full_abs_error_pct_mean', 'mean'),
        n_indices=('index_key', 'nunique'),
    )
    agg = agg.sort_values(['setup_key', 'avg_pure_test_abs_error_pct_mean', 'avg_pure_test_close_RMSE', 'variant_index']).reset_index(drop=True)
    return agg


def build_delta_summary(setup_summary: pd.DataFrame) -> pd.DataFrame:
    if setup_summary.empty:
        return pd.DataFrame()
    base = setup_summary.loc[setup_summary['variant_kind'].eq('base'), ['setup_key', 'avg_pure_test_abs_error_pct_mean', 'avg_pure_test_close_RMSE', 'avg_tuning_abs_error_pct_mean']].rename(columns={'avg_pure_test_abs_error_pct_mean': 'base_pure_test_abs_error_pct_mean', 'avg_pure_test_close_RMSE': 'base_pure_test_close_RMSE', 'avg_tuning_abs_error_pct_mean': 'base_tuning_abs_error_pct_mean'})
    delta = setup_summary.merge(base, on='setup_key', how='left')
    delta['delta_pure_test_abs_error_pct_mean'] = delta['avg_pure_test_abs_error_pct_mean'] - delta['base_pure_test_abs_error_pct_mean']
    delta['delta_pure_test_close_RMSE'] = delta['avg_pure_test_close_RMSE'] - delta['base_pure_test_close_RMSE']
    delta['delta_tuning_abs_error_pct_mean'] = delta['avg_tuning_abs_error_pct_mean'] - delta['base_tuning_abs_error_pct_mean']
    return delta


def plot_delta_bars(delta_summary: pd.DataFrame, out_path: Path) -> None:
    setups = delta_summary['setup_key'].drop_duplicates().tolist()
    fig, axes = plt.subplots(len(setups), 1, figsize=(11.0, 3.0 * len(setups)), squeeze=False)
    for idx, setup_key in enumerate(setups):
        ax = axes[idx][0]
        subset = delta_summary[(delta_summary['setup_key'] == setup_key) & (delta_summary['variant_kind'] == 'oat')].copy()
        ax.bar(subset['variant_label'], subset['delta_pure_test_abs_error_pct_mean'], color='#2F5496')
        ax.axhline(0.0, color='black', linewidth=0.8)
        ax.set_title(f'{setup_key}: delta pure-test abs % error vs base')
        ax.set_ylabel('Delta abs % error')
        ax.tick_params(axis='x', rotation=55)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.close(fig)


In [ ]:

best_setup = load_best_setup()
variant_manifest = build_variant_manifest(best_setup)
case_bundle = load_case_bundle()
variant_manifest.to_csv(VARIANT_MANIFEST_PATH, index=False)

if FORCE_FRESH_RUN:
    reset_output_targets()
    variant_manifest.to_csv(VARIANT_MANIFEST_PATH, index=False)

progress_frame = load_progress_frame()
completed_keys = completed_window_keys(progress_frame)

tasks = []
for case_key, case_info in case_bundle.items():
    window_size = int(case_info['window_size'])
    index_key = case_info['index_key']
    frame = case_info['df'].copy()
    manifest = case_info['window_manifest'].copy()
    for setup_key, setup_block in variant_manifest.groupby('setup_key'):
        model_name = setup_block['model_name'].iloc[0]
        if int(setup_block['window_size'].iloc[0]) != window_size:
            continue
        tasks.append((setup_key, model_name, setup_block['branch_name'].iloc[0], case_key, index_key, frame, manifest, setup_block.copy()))

total_units = 0
completed_units = 0
for setup_key, _, branch_name, case_key, _, _, manifest, setup_block in tasks:
    for _, cfg_row in setup_block.iterrows():
        total_units += len(manifest)
        completed_units += sum(window_result_key(branch_name, setup_key, case_key, int(cfg_row['variant_index']), int(wid)) in completed_keys for wid in manifest['window_id'].tolist())

progress_bar = tqdm(total=total_units, initial=completed_units, desc='DL sensitivity runs', unit='window')
with timed_step('dl_sensitivity_runs'):
    for setup_key, model_name, branch_name, case_key, index_key, frame, manifest, setup_block in tasks:
        branch_log_path = LOG_DIR / f'{setup_key}__{case_key}.csv'
        for _, cfg_row in setup_block.iterrows():
            config = json.loads(cfg_row['config_json'])
            buffer = []
            for _, meta in manifest.iterrows():
                key = window_result_key(branch_name, setup_key, case_key, int(cfg_row['variant_index']), int(meta['window_id']))
                if key in completed_keys:
                    continue
                train_df = frame.iloc[int(meta['start']) : int(meta['start']) + int(meta['train_rows'])].copy()
                test_df = frame.iloc[int(meta['end_exclusive']) - int(meta['test_rows']) : int(meta['end_exclusive'])].copy()
                fit_start = time.perf_counter()
                y_pred = predict_one_window(branch_name, model_name, config, train_df, test_df)
                fit_seconds = time.perf_counter() - fit_start
                row = {
                    'setup_key': setup_key,
                    'branch_name': branch_name,
                    'case_key': case_key,
                    'index_key': index_key,
                    'window_size': int(cfg_row['window_size']),
                    'model_name': model_name,
                    'variant_index': int(cfg_row['variant_index']),
                    'variant_kind': cfg_row['variant_kind'],
                    'variant_label': cfg_row['variant_label'],
                    'changed_param': cfg_row['changed_param'],
                    'changed_to': cfg_row['changed_to'],
                    'config_json': cfg_row['config_json'],
                    'base_config_key': cfg_row['base_config_key'],
                    'window_id': int(meta['window_id']),
                    'train_end_date': meta['train_end_date'],
                    'test_reference_date': meta['test_reference_date'],
                    'test_target_date': meta['test_target_date'],
                    'is_tuning_window': bool(meta['is_tuning_window']),
                    'reference_close': float(test_df['reference_close'].iloc[-1]),
                    'target_close': float(test_df['target_close'].iloc[-1]),
                    'actual_close': float(test_df['target_close'].iloc[-1]),
                    'predicted_close': float(y_pred),
                    'fit_seconds': float(fit_seconds),
                    'status': 'completed',
                    'error_message': '',
                }
                buffer.append(row)
                if len(buffer) >= FLUSH_EVERY_N_ROWS:
                    flush_frame = add_error_columns(pd.DataFrame(buffer))
                    append_frame(ALL_LOGS_PATH, flush_frame)
                    append_frame(branch_log_path, flush_frame)
                    buffer = []
                completed_keys.add(key)
                progress_bar.update(1)
            if buffer:
                flush_frame = add_error_columns(pd.DataFrame(buffer))
                append_frame(ALL_LOGS_PATH, flush_frame)
                append_frame(branch_log_path, flush_frame)
progress_bar.close()


In [ ]:

logs = load_progress_frame()
logs = add_error_columns(logs)
scores_summary = build_scores_summary(logs)
setup_summary = build_setup_summary(scores_summary)
delta_summary = build_delta_summary(setup_summary)

scores_summary.to_csv(SCORES_SUMMARY_PATH, index=False)
setup_summary.to_csv(SETUP_SUMMARY_PATH, index=False)
delta_summary.to_csv(DELTA_SUMMARY_PATH, index=False)
pd.DataFrame(TIMING_ROWS).to_csv(TIMING_SUMMARY_PATH, index=False)

for row in scores_summary.itertuples(index=False):
    subset = logs[(logs['setup_key'] == row.setup_key) & (logs['case_key'] == row.case_key) & (logs['variant_index'] == row.variant_index)].copy().sort_values('window_id')
    pred_path = PREDICTIONS_DIR / f"{row.setup_key}_{row.case_key}_v{row.variant_index:02d}.csv"
    subset.to_csv(pred_path, index=False)

plot_delta_bars(delta_summary, FIG_DIR / '01_delta_pure_test_abs_error.png')
display(delta_summary[['setup_key', 'branch_name', 'variant_index', 'variant_label', 'delta_pure_test_abs_error_pct_mean', 'delta_pure_test_close_RMSE']].head(80))
